# Compass Dashboard

In [8]:
import pandas as pd
import pdfplumber
import sys
import os
sys.path.append(os.path.abspath(".."))


## Preprocessing

Like many mid-sized companies, my groundhandling agency at Vancouver airport loves to distribute employee rosters in the form of pdf documents with schedule laid out in unloveable tables that are neither machine readable nor copy-paste-able. To first extract shift information from these tables, I used the pdfplumber library, which contains a range of functions for dealing with pdf documents.

In [9]:
with pdfplumber.open("../data/Monthly Schedule _ MAR.pdf") as pdf:
    page = pdf.pages[0]
    table = page.extract_table()
pd.DataFrame(table).head()

,0,1,2,3,4,5,6,7
0,MAR 2026,Mon\n02-Mar,Tue\n03-Mar,Wed\n04-Mar,Thu\n05-Mar,Fri\n06-Mar,Sat\n07-Mar,Sun\n08-Mar
1,Ratna\nSalim,,1445-1845\nAGT1,,,1445-1845\nAGT1,,
2,Alex\nYe,,,,,,,
3,Ann\nKim,,1445-1845\nAGT1,1445-1845\nAGT1,,1415-1815\nTKT,,1545-1945\nAGT1
4,Saeko\nSumi,,1215-1915 FC,1215-1915 FC,1215-1915 FC,1215-1915 FC,1215-1915 FC,


So in fact with this specific document it is quite easy to use extract_table and turn it directly into a data frame. However, with PDF documents this is not always the case, and a more robust solution that is good to practice is to extract words with their positions and reconstruct the table using those relative postions. My function below uses pdfplumber's extract words to retrieve all text instances from the table along with their x and y coordinates, and then reconstructs the information regarding my shifts into a dictionary for later use

In [10]:
from src.parser.orchestrator import parse_pdf
data = parse_pdf('../data/Monthly Schedule _ MAR.pdf')
data[:5]

[{'date': '02-Mar', 'time': '1445-1845', 'type': 'AGT1'},
 {'date': '04-Mar', 'time': '1445-1845', 'type': 'AGT1'},
 {'date': '05-Mar', 'time': '1445-1845', 'type': 'AGT1'},
 {'date': '07-Mar', 'time': '1445-1845', 'type': 'AGT1'},
 {'date': '08-Mar', 'time': '1545-1945', 'type': 'AGT1'}]

In [11]:
df = pd.DataFrame(data)
df.head()

,date,time,type
0,02-Mar,1445-1845,AGT1
1,04-Mar,1445-1845,AGT1
2,05-Mar,1445-1845,AGT1
3,07-Mar,1445-1845,AGT1
4,08-Mar,1545-1945,AGT1


This is all the information that I need parsed correctly from the original document, but its current format leaves a bit to be desired. The `clean_dataframe` function below splits the time column into separate start and end times, as well as standardizing the spacing and capitalisation of all values and column names as a way to future-proof the code. It also performs a quick validation of the data and will raise an error if there are any missing dates or time values

In [12]:
from src.processing.cleaning import clean_dataframe

df = clean_dataframe(df)

df.head()

,date,type,start_time,end_time
0,02-Mar,AGT1,1445,1845
1,04-Mar,AGT1,1445,1845
2,05-Mar,AGT1,1445,1845
3,07-Mar,AGT1,1445,1845
4,08-Mar,AGT1,1545,1945


Now the data is in a much more useable form, but the dates are still just strings. An obvious improvement would be to transform them into datetime objects, which is what the `add_datetime_columns` function does. Note that it takes a start year for the schedule and will automatically increment the year in case the date range spans January 1st.

In [13]:
from src.processing.datetime_utils import add_datetime_columns

df = add_datetime_columns(df)

df.head()

,date,type,start_time,end_time
0,2026-03-02,AGT1,1445,1845
1,2026-03-04,AGT1,1445,1845
2,2026-03-05,AGT1,1445,1845
3,2026-03-07,AGT1,1445,1845
4,2026-03-08,AGT1,1545,1945


And from there I can add a few extra features that will be useful when calculating transit costs, using only the datetimes in the first column.

In [14]:
from src.processing.features import add_features

df = add_features(df)

df.head()

,date,type,start_time,end_time,weekday,is_weekend,month
0,2026-03-02,AGT1,1445,1845,Monday,False,March
1,2026-03-04,AGT1,1445,1845,Wednesday,False,March
2,2026-03-05,AGT1,1445,1845,Thursday,False,March
3,2026-03-07,AGT1,1445,1845,Saturday,True,March
4,2026-03-08,AGT1,1545,1945,Sunday,True,March


Wrapping this all together into a pipeline I can create this dataframe with a single function call.

In [15]:
from src.pipeline import run_pipeline

df = run_pipeline('../data/Monthly Schedule _ MAR.pdf', start_year=2026)

df.head()


,date,type,start_time,end_time,weekday,is_weekend,month
0,2026-03-02,AGT1,1445,1845,Monday,False,March
1,2026-03-04,AGT1,1445,1845,Wednesday,False,March
2,2026-03-05,AGT1,1445,1845,Thursday,False,March
3,2026-03-07,AGT1,1445,1845,Saturday,True,March
4,2026-03-08,AGT1,1545,1945,Sunday,True,March


In [16]:
fare_data = {
    '1zone': 2.70,
    '2zone': 4.00,
    '1zonemonth': 111.60,
    '2zonemonth': 149.25,
    '2zoneaddfare': 1.50,
    'yvraddfare' : 5.00
}
fare_data

{'1zone': 2.7,
 '2zone': 4.0,
 '1zonemonth': 111.6,
 '2zonemonth': 149.25,
 '2zoneaddfare': 1.5,
 'yvraddfare': 5.0}

In [17]:
from src.logic.trips import ShiftsToTrips

In [26]:
trips = ShiftsToTrips(df)
trips.head()

,date,month,is_weekend,origin,time
0,2026-03-02,March,False,home,1345
25,2026-03-02,March,False,YVR,1845
26,2026-03-04,March,False,YVR,1845
1,2026-03-04,March,False,home,1345
2,2026-03-05,March,False,home,1345


In [27]:
trips.shape

(50, 5)

In [28]:
def GetOneZoneTrips(df:pd.DataFrame):
    "Find all trips that occur on a weekend or after 6:30pm"
    one_zone = df.loc[(df['is_weekend']==True) | (df['time']>1830)]

    return one_zone

GetOneZoneTrips(trips).head()
GetOneZoneTrips(trips).shape

(35, 5)

In [29]:
def GetTwoZoneTrips(df:pd.DataFrame):
    "find all trips that occur on a weekday before 6:30pm"
    two_zone = df.loc[(df['is_weekend']==False) & (df['time']<1830)]

    return two_zone

GetTwoZoneTrips(trips).shape

(15, 5)

In [34]:
def GetAddFareTrips(df:pd.DataFrame):
    "find all trips that will incur the YVR add fare"
    add_fare = df.loc[df['origin'] == 'YVR']

    return add_fare

GetAddFareTrips(trips).head()

,date,month,is_weekend,origin,time
25,2026-03-02,March,False,YVR,1845
26,2026-03-04,March,False,YVR,1845
27,2026-03-05,March,False,YVR,1845
28,2026-03-07,March,True,YVR,1845
29,2026-03-08,March,True,YVR,1945


In [33]:
GetAddFareTrips(trips).shape

(25, 5)

In [ ]:
def TotalCost(shifts:pd.DataFrame, fares:dict, method:str):
    "find the total cost of commuting based on the given method"

    # convert shifts to trips
    trips = ShiftsToTrips(shifts)

    # count one zone trips
    OneZone = GetOneZoneTrips(trips).shape[0]

    # count two zone trips
    TwoZone = GetTwoZoneTrips(trips).shape[0]

    # count add fare trips
    AddFare = GetAddFareTrips(trips).shape[0]

    if method == 'StoredValue':
        TotalCost = OneZone*fares['1zone'] + TwoZone*fares['2zone'] + AddFare*fares['yvraddfare']
        return TotalCost
    if method == 'OneZoneMonthly':
        TotalCost = fares['1zonemonth'] + TwoZone*fares['2zoneaddfare']
        return TotalCost
    if method == 'TwoZoneMonthly':
        TotalCost = fares['2zonemonth']
        return TotalCost

    return None

TotalCost(shifts, fare_data, 'StoredValue')

33.8

In [ ]:
CommuteCost = {
    'Stored Value' : TotalCost(shifts, fare_data, 'StoredValue'),
    '1 Zone Month Pass' : TotalCost(shifts, fare_data, 'OneZoneMonthly'),
    '2 Zone Month Pass' : TotalCost(shifts, fare_data, 'TwoZoneMonthly')
}

pd.DataFrame(CommuteCost, index=range(1))

,Stored Value,1 Zone Month Pass,2 Zone Month Pass
0,33.8,114.6,149.25
